# Defining the subevent matching algorithm

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx
import matplotlib.cm as cm
import h5py

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch

## Minimal Loading

In [2]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root"
batch = EDM4hepEventBatch(edm_input_file, events=slice(0, 10), full_load=False)

In [4]:
hits_batch = batch.get_tracker_hits_df()
# Print the amount of memory used by the hits_batch
print(f"Memory used by the hits_batch: {hits_batch.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the hits_batch: 302.06 MB


In [5]:
hits_batch

,event_id,subentry,cellID,time,x,y,z,px,py,pz,detector,particle_id,r,R,phi,theta,eta,pt
0,0,0,256188893628166,-4.977191,-31.831104,4.829101,-489.650446,-0.137585,0.191477,0.047277,PixelBarrelReadout,84489,32.195332,490.707753,2.991031,3.075935,-3.416097,0.235782
1,0,1,256802604189702,-4.969477,-32.956036,6.378156,-489.269836,-0.139093,0.190346,0.046564,PixelBarrelReadout,84489,33.567561,490.419977,2.950421,3.073093,-3.373676,0.235751
2,0,2,267388356398358,-4.817920,-57.763868,35.774230,-481.510196,-0.163330,0.166886,0.047070,PixelBarrelReadout,84489,67.944536,486.280299,2.587089,3.001411,-2.656324,0.233511
3,0,3,280237556372774,-4.633086,-93.217879,66.303405,-472.187467,-0.189751,0.134464,0.046297,PixelBarrelReadout,84489,114.392808,485.846394,2.523341,2.903911,-2.125245,0.232564
4,0,4,14086720987190,-4.410011,-142.769602,93.210803,-461.073716,-0.214661,0.087632,0.045286,PixelBarrelReadout,84489,170.503410,491.589651,2.563199,2.787392,-1.720506,0.231859
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2639457,9,30264,16273632309278,16.136225,666.254177,-453.583257,1579.500000,-0.002421,0.003459,-0.002720,LongStripEndcapReadout,1024851,805.997766,1773.260457,-0.597721,0.471843,1.425458,0.004223
2639458,9,30265,16389596426270,16.215624,665.877732,-458.224097,1579.375000,-0.000123,0.000597,0.000262,LongStripEndcapReadout,1024851,808.308405,1774.200628,-0.602729,0.473035,1.422839,0.000609
2639459,9,30266,16381006491678,16.217428,665.881585,-457.851227,1579.375126,-0.000042,0.000637,-0.000117,LongStripEndcapReadout,1024851,808.100260,1774.105921,-0.602346,0.472931,1.423068,0.000638
2639460,9,30267,1602023927870,14.639566,-652.289036,569.961386,2229.500988,-0.000083,-0.000062,-0.000058,LongStripEndcapReadout,1024954,866.219930,2391.863629,2.423450,0.370576,1.674306,0.000104


In [6]:
particles_batch = batch.get_particles_df()
# Print the amount of memory used by the particles_batch
print(f"Memory used by the particles_batch: {particles_batch.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the particles_batch: 1336.18 MB


In [ ]:
particles_batch

,event_id,subentry,PDG,generatorStatus,simulatorStatus,charge,time,mass,vx,vy,...,daughters_begin,daughters_end,vr,endpoint_r,energy,kinetic_energy,particle_id,created_in_simulation,num_tracker_hits,num_calo_hits
0,0,0,2212,4,0,1.000000,0.000000,0.938270,0.000000,0.000000,...,0,4,0.000000,0.010252,7000.000000,6999.061730,0,False,0,0
1,0,1,-2,61,0,-0.666667,-0.765510,0.000000,0.009780,-0.003074,...,4,5,0.010252,0.010252,1028.604778,1028.604778,1,False,0,0
2,0,2,2,63,0,0.666667,-0.765510,0.330000,0.009780,-0.003074,...,5,6,0.010252,0.010252,2208.906631,2208.576631,2,False,0,0
3,0,3,2103,63,0,0.333333,-0.765510,0.771330,0.009780,-0.003074,...,6,7,0.010252,0.010252,1147.773510,1147.002180,3,False,0,0
4,0,4,2,63,0,0.666667,-0.765510,0.330000,0.009780,-0.003074,...,7,8,0.010252,0.010252,2614.671253,2614.341253,4,False,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9662671,9,1024983,11,0,1426063360,-1.000000,9.526830,0.000511,664.288709,1095.734946,...,1211678,1211678,1281.372140,1285.722079,0.016720,0.016209,1024983,True,0,0
9662672,9,1024984,-11,0,1426063360,1.000000,9.526830,0.000511,664.288709,1095.734946,...,1211678,1211679,1281.372140,1285.017953,0.010943,0.010432,1024984,True,0,0
9662673,9,1024985,11,0,1426063360,-1.000000,9.650167,0.000511,674.322801,1124.913154,...,1211679,1211679,1311.541400,1312.963191,0.002300,0.001789,1024985,True,0,0
9662674,9,1024986,11,0,1426063360,-1.000000,9.597322,0.000511,676.994870,1102.626842,...,1211679,1211679,1293.873258,1294.608567,0.002240,0.001729,1024986,True,0,0


In [7]:
calo_hits_batch = batch.get_calo_hits_df()
# Print the amount of memory used by the calo_hits_batch
print(f"Memory used by the calo_hits_batch: {calo_hits_batch.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the calo_hits_batch: 1081.14 MB


In [8]:
calo_hits_batch

,event_id,subentry,cellID,energy,x,y,z,contribution_begin,contribution_end,detector,r,R,phi,theta,eta
0,0,0,104145539527933968,5.172027e-04,719.657898,-1058.644897,1881.900024,0,14,ECalBarrelCollection,1280.092407,2276.001709,-0.973762,0.597322,1.178078
1,0,1,50102494323496976,1.691220e-06,545.476929,1156.974487,902.700012,14,15,ECalBarrelCollection,1279.114990,1565.567749,1.130233,0.956240,0.657348
2,0,2,18278703426723737616,1.027635e-07,1459.449951,-102.000000,-3049.800049,15,16,ECalBarrelCollection,1463.010010,3382.554932,-0.069776,2.694311,-1.480845
3,0,3,18577387126204432,5.202330e-05,-1162.829468,-531.341553,336.600006,16,17,ECalBarrelCollection,1278.474243,1322.042236,-2.712984,1.313356,0.260332
4,0,4,18577387126237200,5.268137e-05,-1167.495117,-533.274109,336.600006,17,18,ECalBarrelCollection,1283.520996,1326.923462,-2.713125,1.314324,0.259331
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13495871,9,238371,3377966015119636,7.426408e-08,1754.028076,715.950684,4106.500000,1461553,1461555,HCalEndcapCollection,1894.518433,4522.448730,0.387534,0.432249,1.516158
13495872,9,238372,1970621196173588,9.218313e-07,1989.256592,609.801880,3851.500000,1461555,1461560,HCalEndcapCollection,2080.625000,4377.562500,0.297453,0.495297,1.375004
13495873,9,238373,1689146219462932,7.732384e-08,1995.109253,580.378296,3851.500000,1461560,1461561,HCalEndcapCollection,2077.811279,4376.225586,0.283088,0.494731,1.376195
13495874,9,238374,2533566854594836,1.466863e-10,1948.127563,662.796265,3800.500000,1461561,1461562,HCalEndcapCollection,2057.790039,4321.839844,0.327938,0.496257,1.372986


In [9]:
calo_contributions_batch = batch.get_calo_contributions_df()
# Print the amount of memory used by the calo_contributions_batch
print(f"Memory used by the calo_contributions_batch: {calo_contributions_batch.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the calo_contributions_batch: 2277.50 MB


In [10]:
calo_contributions_batch

,event_id,subentry,energy,time,particle_id,detector
0,0,0,6.181428e-05,367.544739,84496,ECalBarrelCollection
1,0,1,9.740763e-05,367.545288,84496,ECalBarrelCollection
2,0,2,7.868026e-06,367.551208,84496,ECalBarrelCollection
3,0,3,7.590641e-06,367.551300,84496,ECalBarrelCollection
4,0,4,5.001049e-06,367.551422,84496,ECalBarrelCollection
...,...,...,...,...,...,...
66336969,9,1461560,7.732384e-08,523.905640,1024769,HCalEndcapCollection
66336970,9,1461561,1.466863e-10,4218.285156,1024769,HCalEndcapCollection
66336971,9,1461562,1.033361e-08,4700.229492,1024769,HCalEndcapCollection
66336972,9,1461563,1.716136e-09,4735.110352,1024769,HCalEndcapCollection


## Eventwise

In [2]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root"
event = EDM4hepEvent(edm_input_file, event_index=0)

Loading event 0 from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root...
  Loaded 880327 particles.
  Loaded 240622 tracker hits.
  Loaded 1243953 calo hits and 6038426 contributions.


In [3]:
hits = event.get_tracker_hits_df()

In [4]:
calo_hits = event.get_calo_contributions_df()

In [5]:
particles = event.get_particles_df()

In [6]:
%%time

for i in range(10):
    event = EDM4hepEvent(edm_input_file, event_index=i)
    hits = event.get_tracker_hits_df()

CPU times: user 4 μs, sys: 1 μs, total: 5 μs
Wall time: 10.7 μs
Loading event 0 from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root...
  Loaded 880327 particles.
  Loaded 240622 tracker hits.
  Loaded 1243953 calo hits and 6038426 contributions.
Loading event 1 from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root...
  Loaded 948682 particles.
  Loaded 268168 tracker hits.
  Loaded 1330005 calo hits and 6470534 contributions.
Loading event 2 from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root...
  Loaded 949723 particles.
  Loaded 252857 tracker hits.
  Loaded 1329280 calo hits and 6537286 contributions.
Loading event 3 from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root...
  Loaded 946346 particles.
  Loaded 251185 tracker hits.
  Loaded 1309217 calo hits and 6497732 contributions.
Loading event 4 from /ps

## Batch loading

In [13]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root"
batch = EDM4hepEventBatch(edm_input_file, events=slice(0, 10))

In [14]:
hits_batch = batch.get_tracker_hits_df()
# Print the amount of memory used by the hits_batch
print(f"Memory used by the hits_batch: {hits_batch.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the hits_batch: 332.27 MB


In [15]:
hits_batch

,event_id,subentry,cellID,time,pathLength,quality,x,y,z,px,...,pz,EDep,detector,particle_id,r,R,phi,theta,eta,pt
0,0,0,256188893628166,-4.977191,0.163255,0,-31.831104,4.829101,-489.650446,-0.137585,...,0.047277,0.000059,PixelBarrelReadout,84489,32.195332,490.707753,2.991031,3.075935,-3.416097,0.235782
1,0,1,256802604189702,-4.969477,0.259380,0,-32.956036,6.378156,-489.269836,-0.139093,...,0.046564,0.000079,PixelBarrelReadout,84489,33.567561,490.419977,2.950421,3.073093,-3.373676,0.235751
2,0,2,267388356398358,-4.817920,0.134571,0,-57.763868,35.774230,-481.510196,-0.163330,...,0.047070,0.000053,PixelBarrelReadout,84489,67.944536,486.280299,2.587089,3.001411,-2.656324,0.233511
3,0,3,280237556372774,-4.633086,0.128650,0,-93.217879,66.303405,-472.187467,-0.189751,...,0.046297,0.000044,PixelBarrelReadout,84489,114.392808,485.846394,2.523341,2.903911,-2.125245,0.232564
4,0,4,14086720987190,-4.410011,0.127594,0,-142.769602,93.210803,-461.073716,-0.214661,...,0.045286,0.000041,PixelBarrelReadout,84489,170.503410,491.589651,2.563199,2.787392,-1.720506,0.231859
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2639457,9,30264,16273632309278,16.136225,0.430938,0,666.254177,-453.583257,1579.500000,-0.002421,...,-0.002720,0.000104,LongStripEndcapReadout,1024851,805.997766,1773.260457,-0.597721,0.471843,1.425458,0.004223
2639458,9,30265,16389596426270,16.215624,0.323981,0,665.877732,-458.224097,1579.375000,-0.000123,...,0.000262,0.000163,LongStripEndcapReadout,1024851,808.308405,1774.200628,-0.602729,0.473035,1.422839,0.000609
2639459,9,30266,16381006491678,16.217428,0.055901,0,665.881585,-457.851227,1579.375126,-0.000042,...,-0.000117,0.000025,LongStripEndcapReadout,1024851,808.100260,1774.105921,-0.602346,0.472931,1.423068,0.000638
2639460,9,30267,1602023927870,14.639566,0.026212,0,-652.289036,569.961386,2229.500988,-0.000083,...,-0.000058,0.000053,LongStripEndcapReadout,1024954,866.219930,2391.863629,2.423450,0.370576,1.674306,0.000104


In [16]:
particles_batch = batch.get_particles_df()
# Print the amount of memory used by the particles_batch
print(f"Memory used by the particles_batch: {particles_batch.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the particles_batch: 1741.64 MB


In [17]:
particles_batch

,event_id,subentry,PDG,generatorStatus,simulatorStatus,charge,time,mass,vx,vy,...,daughters_begin,daughters_end,vr,endpoint_r,energy,kinetic_energy,particle_id,created_in_simulation,num_tracker_hits,num_calo_hits
0,0,0,2212,4,0,1.000000,0.000000,0.938270,0.000000,0.000000,...,0,4,0.000000,0.010252,7000.000000,6999.061730,0,False,0,0
1,0,1,-2,61,0,-0.666667,-0.765510,0.000000,0.009780,-0.003074,...,4,5,0.010252,0.010252,1028.604778,1028.604778,1,False,0,0
2,0,2,2,63,0,0.666667,-0.765510,0.330000,0.009780,-0.003074,...,5,6,0.010252,0.010252,2208.906631,2208.576631,2,False,0,0
3,0,3,2103,63,0,0.333333,-0.765510,0.771330,0.009780,-0.003074,...,6,7,0.010252,0.010252,1147.773510,1147.002180,3,False,0,0
4,0,4,2,63,0,0.666667,-0.765510,0.330000,0.009780,-0.003074,...,7,8,0.010252,0.010252,2614.671253,2614.341253,4,False,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9662671,9,1024983,11,0,1426063360,-1.000000,9.526830,0.000511,664.288709,1095.734946,...,1211678,1211678,1281.372140,1285.722079,0.016720,0.016209,1024983,True,0,0
9662672,9,1024984,-11,0,1426063360,1.000000,9.526830,0.000511,664.288709,1095.734946,...,1211678,1211679,1281.372140,1285.017953,0.010943,0.010432,1024984,True,0,0
9662673,9,1024985,11,0,1426063360,-1.000000,9.650167,0.000511,674.322801,1124.913154,...,1211679,1211679,1311.541400,1312.963191,0.002300,0.001789,1024985,True,0,0
9662674,9,1024986,11,0,1426063360,-1.000000,9.597322,0.000511,676.994870,1102.626842,...,1211679,1211679,1293.873258,1294.608567,0.002240,0.001729,1024986,True,0,0


In [18]:
calo_hits_batch = batch.get_calo_hits_df()
# Print the amount of memory used by the calo_hits_batch
print(f"Memory used by the calo_hits_batch: {calo_hits_batch.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the calo_hits_batch: 1081.14 MB


In [19]:
calo_hits_batch

,event_id,subentry,cellID,energy,x,y,z,contribution_begin,contribution_end,detector,r,R,phi,theta,eta
0,0,0,104145539527933968,5.172027e-04,719.657898,-1058.644897,1881.900024,0,14,ECalBarrelCollection,1280.092407,2276.001709,-0.973762,0.597322,1.178078
1,0,1,50102494323496976,1.691220e-06,545.476929,1156.974487,902.700012,14,15,ECalBarrelCollection,1279.114990,1565.567749,1.130233,0.956240,0.657348
2,0,2,18278703426723737616,1.027635e-07,1459.449951,-102.000000,-3049.800049,15,16,ECalBarrelCollection,1463.010010,3382.554932,-0.069776,2.694311,-1.480845
3,0,3,18577387126204432,5.202330e-05,-1162.829468,-531.341553,336.600006,16,17,ECalBarrelCollection,1278.474243,1322.042236,-2.712984,1.313356,0.260332
4,0,4,18577387126237200,5.268137e-05,-1167.495117,-533.274109,336.600006,17,18,ECalBarrelCollection,1283.520996,1326.923462,-2.713125,1.314324,0.259331
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13495871,9,238371,3377966015119636,7.426408e-08,1754.028076,715.950684,4106.500000,1461553,1461555,HCalEndcapCollection,1894.518433,4522.448730,0.387534,0.432249,1.516158
13495872,9,238372,1970621196173588,9.218313e-07,1989.256592,609.801880,3851.500000,1461555,1461560,HCalEndcapCollection,2080.625000,4377.562500,0.297453,0.495297,1.375004
13495873,9,238373,1689146219462932,7.732384e-08,1995.109253,580.378296,3851.500000,1461560,1461561,HCalEndcapCollection,2077.811279,4376.225586,0.283088,0.494731,1.376195
13495874,9,238374,2533566854594836,1.466863e-10,1948.127563,662.796265,3800.500000,1461561,1461562,HCalEndcapCollection,2057.790039,4321.839844,0.327938,0.496257,1.372986


In [20]:
calo_contributions_batch = batch.get_calo_contributions_df()
# Print the amount of memory used by the calo_contributions_batch
print(f"Memory used by the calo_contributions_batch: {calo_contributions_batch.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the calo_contributions_batch: 3289.72 MB


In [21]:
calo_contributions_batch

,event_id,subentry,PDG,energy,time,step_x,step_y,step_z,particle_id,detector
0,0,0,0,6.181428e-05,367.544739,0.0,0.0,0.0,84496,ECalBarrelCollection
1,0,1,0,9.740763e-05,367.545288,0.0,0.0,0.0,84496,ECalBarrelCollection
2,0,2,0,7.868026e-06,367.551208,0.0,0.0,0.0,84496,ECalBarrelCollection
3,0,3,0,7.590641e-06,367.551300,0.0,0.0,0.0,84496,ECalBarrelCollection
4,0,4,0,5.001049e-06,367.551422,0.0,0.0,0.0,84496,ECalBarrelCollection
...,...,...,...,...,...,...,...,...,...,...
66336969,9,1461560,0,7.732384e-08,523.905640,0.0,0.0,0.0,1024769,HCalEndcapCollection
66336970,9,1461561,0,1.466863e-10,4218.285156,0.0,0.0,0.0,1024769,HCalEndcapCollection
66336971,9,1461562,0,1.033361e-08,4700.229492,0.0,0.0,0.0,1024769,HCalEndcapCollection
66336972,9,1461563,0,1.716136e-09,4735.110352,0.0,0.0,0.0,1024769,HCalEndcapCollection


In [3]:
import pandas as pd
import numpy as np
import awkward as ak

def load_root_file(
    file_path, 
    event_offset=0, 
    event_id=None, 
    ignore_variable_columns=True,
    included_columns=None,
    ):
    """Load data from a single root file with optional event filtering
    
    Parameters:
    -----------
    file_path : Path or str
        Path to the root file
    event_offset : int
        Offset to add to event_id (for backwards compatibility)
    event_id : int, optional
        If provided, only load this specific event
    ignore_variable_columns : bool
        Whether to ignore variable length columns
    included_columns : list, optional
        If provided, only load these specific columns
    
    Returns:
    --------
    pd.DataFrame or None
        DataFrame containing the loaded data, or None if loading fails
    """
    import time
    start_time = time.time()
    
    try:
        # Step 1: Open file
        step_start = time.time()
        tree = uproot.open(file_path)
        print(f"File opening took: {time.time() - step_start:.3f}s")
        
        # Step 2: Get keys and find latest
        step_start = time.time()
        keys = tree.keys()
        cycles = [int(key.split(';')[1]) for key in keys]
        latest_key = keys[cycles.index(max(cycles))]
        print(f"Key processing took: {time.time() - step_start:.3f}s")
        
        # Step 3: Determine filtering parameters
        step_start = time.time()
        filter_params = {}
        
        # Column filtering
        if included_columns:
            filter_params['filter_name'] = included_columns
        elif ignore_variable_columns:
            def exclude_var_columns(branch_name):
                branch = tree[latest_key][branch_name]
                return 'var' not in str(branch.typename)
            filter_params['filter_name'] = exclude_var_columns
        
        # Event filtering - this is the key optimization!
        if event_id is not None:
            # Check if we have event_id or event_nr column to determine which to filter on
            available_branches = tree[latest_key].keys()
            if 'event_id' in available_branches:
                # Adjust for event_offset if needed
                target_event = event_id - event_offset if event_offset else event_id
                filter_params['cut'] = f"event_id == {target_event}"
            elif 'event_nr' in available_branches:
                filter_params['cut'] = f"event_nr == {event_id}"
            else:
                print("Warning: No event_id or event_nr column found for event filtering")
        
        print(f"Filter preparation took: {time.time() - step_start:.3f}s")
        
        # Step 4: Load arrays with all filtering applied at once
        step_start = time.time()
        data = tree[latest_key].arrays(**filter_params)
        print(f"Array loading took: {time.time() - step_start:.3f}s")
        
        # Check if we got any data when filtering for specific event
        if event_id is not None and len(data) == 0:
            print(f"No data found for event_id {event_id}")
            return None

        # Step 5: Convert to dataframe 
        step_start = time.time()
        df = ak.to_dataframe(data)
        print(f"DataFrame conversion took: {time.time() - step_start:.3f}s")
            
        # Step 6: Apply event offset (only if we're not filtering to specific event)
        step_start = time.time()
        if event_offset and event_id is None and 'event_id' in df.columns:
            df['event_id'] += event_offset
        print(f"Event offset application took: {time.time() - step_start:.3f}s")
                
        print(f"Total loading time: {time.time() - start_time:.3f}s")
        return df
    
    except (FileNotFoundError, uproot.exceptions.KeyInFileError) as e:
        print(f"Error loading {file_path}: {str(e)}")
        return None

In [4]:
measurements_file_path = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/measurements.root"
included_columns = [
    "event_nr",
    "volume_id", "layer_id", "surface_id",
    "rec_x", "rec_y", "rec_z",
    "true_x", "true_y", "true_z",
]
measurements_df = load_root_file(measurements_file_path, included_columns=included_columns)
# Print the amount of memory used by the measurements_df
print(f"Memory used by the measurements_df: {measurements_df.memory_usage().sum() / 1024 / 1024:.2f} MB")

File opening took: 0.029s
Key processing took: 0.000s
Filter preparation took: 0.000s
Array loading took: 6.267s
DataFrame conversion took: 8.282s
Event offset application took: 0.000s
Total loading time: 14.579s
Memory used by the measurements_df: 1206.32 MB


In [5]:
measurements_df

,event_nr,volume_id,layer_id,surface_id,rec_x,rec_y,rec_z,true_x,true_y,true_z
entry,,,,,,,,,,
0,35,16,4,1,63.634239,-8.141245,-1515.599976,63.643333,-8.139078,-1515.599976
1,35,16,4,1,87.576599,-3.831904,-1515.599976,87.585510,-3.821328,-1515.599976
2,35,16,4,1,80.723595,1.915017,-1515.599976,80.713394,1.892023,-1515.599976
3,35,16,4,1,80.406219,6.460104,-1515.599976,80.424278,6.472737,-1515.599976
4,35,16,4,1,80.437347,6.483580,-1515.599976,80.445496,6.482724,-1515.637817
...,...,...,...,...,...,...,...,...,...,...
31623017,127,30,12,192,413.361084,901.150452,3009.500000,431.460754,947.641357,3009.473145
31623018,127,30,12,192,320.925629,937.290100,3009.500000,344.446716,997.434937,3009.500000
31623019,127,30,12,192,435.983429,892.305786,3009.500000,435.761719,891.883789,3009.617920


In [6]:
particles_file_path = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root"


In [7]:
included_columns = [
    "event_id",
    "vx", "vy", "vz",
    "px", "py", "pz",
    "vertex_primary",
]
particles_df = load_root_file(particles_file_path, event_id=0, included_columns=included_columns)
# Print the amount of memory used by the particles_df
print(f"Memory used by the particles_df: {particles_df.memory_usage().sum() / 1024 / 1024:.2f} MB")
particles_df

File opening took: 0.019s
Key processing took: 0.000s
Filter preparation took: 0.017s
Array loading took: 22.386s
DataFrame conversion took: 0.279s
Event offset application took: 0.000s
Total loading time: 22.701s
Memory used by the particles_df: 68.33 MB


event_id           vx          vy          vz         px  \
entry subentry                                                             
0     0                0     0.009780   -0.003074  -89.848335  19.532045   
      1                0     0.009780   -0.003074  -89.848335   2.695038   
      2                0     0.009780   -0.003074  -89.848335   1.519203   
      3                0     0.009780   -0.003074  -89.848335   0.241998   
      4                0     0.009780   -0.003074  -89.848335   0.930625   
...                  ...          ...         ...         ...        ...   
      840815           0  -246.585052 -265.515167    1.465524  -0.000014   
      840816           0 -1168.682739 -454.746796  341.894836   0.030357   
      840817           0  -474.002075  -10.628981  417.302246   0.035830   
      840818           0     0.004192   -0.000449   16.953886  -0.009112   
      840819           0     0.004192   -0.000449   16.953886   0.055324   

                       py         pz  vertex_primary  
entry subentry                                        
0     0         37.186535  72.377266               1  
      1          4.369767   8.390165               1  
      2          1.424487   4.669379               1  
      3          0.348130   2.346750               1  
      4         -0.356436   3.041831               1  
...                   ...        ...             ...  
      840815     0.000045  -0.000102             201  
      840816    -0.062274  -0.010156             201  
      840817     0.041002  -0.004408             201  
      840818     0.054969  -7.333537             201  
      840819     0.014981  -1.315810             201  

[840820 rows x 8 columns]

In [6]:
import uproot

In [7]:
particles_root_file = uproot.open(particles_file_path)
particles_root_file.keys(), particles_root_file["particles"].keys()

(['particles;15', 'particles;14'],
 ['event_id',
  'particle_id',
  'particle_type',
  'process',
  'vx',
  'vy',
  'vz',
  'vt',
  'px',
  'py',
  'pz',
  'm',
  'q',
  'eta',
  'phi',
  'pt',
  'p',
  'vertex_primary',
  'vertex_secondary',
  'particle',
  'generation',
  'sub_particle',
  'e_loss',
  'total_x0',
  'total_l0',
  'number_of_hits',
  'outcome'])

In [1]:
included_columns = [
    "event_id",
    "vx", "vy", "vz",
    "px", "py", "pz",
    "vertex_primary",
]


In [22]:
particle_arrays = particles_root_file["particles"].arrays(cut="event_id <= 100", filter_name=included_columns)

In [14]:
particle_root_df = ak.to_dataframe(particle_arrays)
# print the amount of memory used by the particle_root_df
print(f"Memory used by the particle_root_df: {particle_root_df.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the particle_root_df: 357.14 MB


In [16]:
particle_root_df = ak.to_dataframe(particle_arrays)
# print the amount of memory used by the particle_root_df
print(f"Memory used by the particle_root_df: {particle_root_df.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the particle_root_df: 637.89 MB


In [18]:
particle_root_df = ak.to_dataframe(particle_arrays)
# print the amount of memory used by the particle_root_df
print(f"Memory used by the particle_root_df: {particle_root_df.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the particle_root_df: 910.81 MB


In [21]:
particle_root_df = ak.to_dataframe(particle_arrays)
# print the amount of memory used by the particle_root_df
print(f"Memory used by the particle_root_df: {particle_root_df.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the particle_root_df: 1177.83 MB


In [23]:
particle_root_df = ak.to_dataframe(particle_arrays)
# print the amount of memory used by the particle_root_df
print(f"Memory used by the particle_root_df: {particle_root_df.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the particle_root_df: 2783.69 MB


In [24]:
particle_root_df

vx          vy           vz         px         py  \
entry subentry                                                              
0     0           0.007940    0.011831    42.213245   7.186532 -10.795485   
      1           0.007940    0.011831    42.213245   3.130744  -4.916332   
      2           0.007940    0.011831    42.213245   1.927759  -2.989460   
      3           0.007940    0.011831    42.213245  11.268869 -12.789291   
      4           0.007940    0.011831    42.213245   6.546195  -8.508306   
...                    ...         ...          ...        ...        ...   
100   834448    607.507446  702.452087 -3218.021240   0.690029  -0.237202   
      834449      0.005816   -0.062600  -665.233215   0.204533   0.277762   
      834450      0.005816   -0.062600  -665.233215   0.077278   0.038777   
      834451      0.001580    0.001990   -64.485588   0.049831  -0.014403   
      834452      0.001580    0.001990   -64.485588  -0.043170  -0.098944   

                        pz  vertex_primary  
entry subentry                              
0     0           6.089607               1  
      1           2.898874               1  
      2           1.591128               1  
      3           5.732779               1  
      4           4.248754               1  
...                    ...             ...  
100   834448     -0.311431             201  
      834449    -69.503693             201  
      834450     -9.398432             201  
      834451   -145.472702             201  
      834452   -790.657471             201  

[87173022 rows x 7 columns]

## Size Testing

Currently 2.2GB

In [ ]:
h5_file = h5py.File("/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/reco/tracker_hits/full_pileup_pilot.ttbar.v2.reco.tracker_hits.events0-99.h5", "r")

In [23]:
import pandas as pd
pd.DataFrame(h5_file["events"]["event_0"]["measurements"][:])

,x,y,z,volume_id,layer_id,surface_id,true_x,true_y,true_z,time,px,py,pz,particle_id,cell_id,detector,e_dep,path_length
0,85.607491,3.613425,-1515.599976,16,4,1,85.631775,3.625161,-1515.599976,35.864868,-0.127980,0.013338,-0.021521,703493,67494740071,1,0.000370,0.821243
1,92.881134,-3.039804,-1515.599976,16,4,1,92.851669,-3.027657,-1515.599976,13.531095,0.165640,-0.054582,-2.974995,67930,1023410279,1,0.000036,0.125214
2,66.319473,-7.858758,-1515.599976,16,4,1,66.317802,-7.857886,-1515.599976,9.396918,0.268940,0.010043,-6.876966,74845,2634023015,1,0.000029,0.125096
3,65.872543,-3.477791,-1515.599976,16,4,1,65.890350,-3.451361,-1515.599976,8655.912109,-0.011574,-0.037811,-0.014304,669642,1157628007,1,0.000116,0.369049
4,92.184578,-4.563632,-1515.599976,16,4,1,92.188545,-4.542456,-1515.599976,6.495801,-0.001369,0.025355,-0.071781,375396,1526726759,1,0.000101,0.132867
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240609,406.731476,903.742432,3009.500000,30,12,192,421.881561,942.290344,3009.500000,7.946029,0.693225,0.425183,2.162005,77697,858994544990,5,0.000081,0.267095
240610,325.683228,935.429993,3009.500000,30,12,192,306.645935,886.782227,3009.500000,11.627073,0.005575,0.012120,-0.007825,343955,15955804590430,5,0.000144,0.522783
240611,353.611938,924.510681,3009.500000,30,12,192,331.738159,868.459473,3009.500000,10.063175,0.001385,0.003332,-0.001101,573373,16819093016926,5,0.000224,0.663010
240612,435.766296,892.390686,3009.500000,30,12,192,446.899231,921.189087,3009.500000,7.529224,-0.001016,-0.001745,0.001744,759736,1748052775262,5,0.000116,0.420584


In [25]:
import pandas as pd
pd.DataFrame(h5_file["events"]["event_0"]["measurements"][:])

,x,y,z,volume_id,layer_id,surface_id,true_x,true_y,true_z,time,px,py,pz,particle_id,cell_id,detector,e_dep,path_length
0,85.607491,3.613425,-1515.599976,16,4,1,85.631775,3.625161,-1515.599976,35.864868,-0.127980,0.013338,-0.021521,703493,67494740071,1,0.000370,0.821243
1,92.881134,-3.039804,-1515.599976,16,4,1,92.851669,-3.027657,-1515.599976,13.531095,0.165640,-0.054582,-2.974995,67930,1023410279,1,0.000036,0.125214
2,66.319473,-7.858758,-1515.599976,16,4,1,66.317802,-7.857886,-1515.599976,9.396918,0.268940,0.010043,-6.876966,74845,2634023015,1,0.000029,0.125096
3,65.872543,-3.477791,-1515.599976,16,4,1,65.890350,-3.451361,-1515.599976,8655.912109,-0.011574,-0.037811,-0.014304,669642,1157628007,1,0.000116,0.369049
4,92.184578,-4.563632,-1515.599976,16,4,1,92.188545,-4.542456,-1515.599976,6.495801,-0.001369,0.025355,-0.071781,375396,1526726759,1,0.000101,0.132867
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240609,406.731476,903.742432,3009.500000,30,12,192,421.881561,942.290344,3009.500000,7.946029,0.693225,0.425183,2.162005,77697,858994544990,5,0.000081,0.267095
240610,325.683228,935.429993,3009.500000,30,12,192,306.645935,886.782227,3009.500000,11.627073,0.005575,0.012120,-0.007825,343955,15955804590430,5,0.000144,0.522783
240611,353.611938,924.510681,3009.500000,30,12,192,331.738159,868.459473,3009.500000,10.063175,0.001385,0.003332,-0.001101,573373,16819093016926,5,0.000224,0.663010
240612,435.766296,892.390686,3009.500000,30,12,192,446.899231,921.189087,3009.500000,7.529224,-0.001016,-0.001745,0.001744,759736,1748052775262,5,0.000116,0.420584


In [6]:
h5_particles = h5py.File("/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/truth/particles/full_pileup_pilot.ttbar.v2.truth.particles.events0-99.h5", "r")

In [8]:
pd.DataFrame(h5_particles["events"]["event_0"]["particles"][:])

,particle_id,pdg_id,mass,energy,charge,vx,vy,vz,time,px,py,pz,vertex_primary
0,0,2212,0.938270,7000.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,7000.000000,NaN
1,1,-2,0.000000,1028.604778,-0.666667,0.009780,-0.003074,-89.848335,-0.765510,-1.615587e-27,0.000000,1028.604736,NaN
2,2,2,0.330000,2208.906631,0.666667,0.009780,-0.003074,-89.848335,-0.765510,0.000000e+00,0.000000,2208.906494,NaN
3,3,2103,0.771330,1147.773510,0.333333,0.009780,-0.003074,-89.848335,-0.765510,0.000000e+00,0.000000,1147.773193,NaN
4,4,2,0.330000,2614.671253,0.666667,0.009780,-0.003074,-89.848335,-0.765510,0.000000e+00,0.000000,2614.671143,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
880324,880322,-11,0.000511,0.007907,1.000000,-72.673141,1323.954956,2621.724854,9.437787,-2.477711e-03,0.002932,0.006894,1.0
880325,880323,-11,0.000511,0.029932,1.000000,-64.631561,1297.143433,2596.949951,9.349813,9.149698e-03,-0.028324,0.003121,1.0
880326,880324,11,0.000511,0.002381,-1.000000,1249.222534,-386.984009,2908.253662,8859.554688,1.222434e-03,-0.001350,-0.001447,1.0
880327,880325,2112,0.939565,0.940569,0.000000,1289.538086,400.301910,2893.576172,25.586529,2.374915e-02,0.035455,0.008157,1.0
